In [97]:
# Headstone_Card_Images.py
# 5/5/2026
# Reads the card images, extract Next of Kin information and stores NOK information

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys

import random
import requests
import math
import numpy as np
import urllib3
import inspect
from pathlib import Path
import json
import base64
import mimetypes
from openai import OpenAI
import os
from PIL import Image
from IPython.display import display


def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print('\n')
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
        print('\n')
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def get_tic():
    tic=datetime.datetime.now()
    return tic




# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++
blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Headstone Next Of Kin (NOK)'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++

# +++++++++++++++++++ File Input and Output +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\0-Master Worksheets\\Ancestry\\"

fileInput="Headstone_Images_X"
fileOutput="Headstone_Details"



fileIn=DirMain+"Inputs\\"+fileInput+".csv"
fileOut=DirMain+"Outputs\\"+fileOutput+"_"+timestamp+".csv"
fileOutInt=DirMain+"Outputs\\"+fileOutput+"_"+timestamp+"_Interim.csv"
fileOutCrop=DirMain+"Headstone_Cropped_Images\\"

tic_start_total=get_tic()
tic_start=get_tic()
tic=get_tic()  




def crop_bottom_of_page(image_path, crop_percent=0.3):
    image_path = Path(image_path)
    img = Image.open(image_path)
    width, height = img.size
    # Keep only bottom 35% of page
    top = int(height * (1 - crop_percent))
    bottom = height
    cropped = img.crop((0, top, width, bottom))
    cropped_path = image_path.parent / f"{image_path.stem}_bottom{image_path.suffix}"
    crop_image_path=fileOutCrop+ Path(image_path).stem+'_Crop.jpg'
    cropped.save(cropped_path)
    cropped.save(crop_image_path)
    # display(cropped)
    return cropped_path
    
def image_to_data_url(image_path):
    image_path = Path(image_path)
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/jpeg"
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"

def extract_next_of_kin(image_path):
    data_url = image_to_data_url(image_path)
    prompt = """
    You are reading a U.S. military Headstone Inscription and Interment Record.
    
    Extract ONLY the "Name and Address of Next of Kin" field.
    
    Return valid JSON only.
    
    Use this exact structure:
    {
      "next_of_kin_name": "",
      "relationship": "",
      "street_address": "",
      "city": "",
      "state": "",
      "raw_next_of_kin_text": "",
      "uncertain": false
    }
    
    Rules:
    - Do not extract the soldier's name.
    - Only use the Next of Kin box.
    - Convert state to standard 2-letter USPS abbreviation.
    - If the relationship is in parentheses, put it in "relationship".
    - If a field is not visible, use an empty string.
    - If unsure, set "uncertain": true.
    """
    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text":prompt},
                    {
                        "type": "input_image",
                        "image_url": data_url,
                        "detail": "high"
                    }
                ],
            }
        ],
    )
    usage = response.usage
    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens
    total_tokens = usage.total_tokens    

    # print("Got response from OpenAI")
    text = response.output_text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    result=json.loads(text)
    result["input_tokens"] = input_tokens
    result["output_tokens"] = output_tokens
    result["total_tokens"] = total_tokens
    return result    



NOK_input_df = pd.read_csv(INPUT_CSV)
# total_names=len(NOK_input_df)
# rows = []
# NOK_df=pd.DataFrame([])
# cum_total_tokens=0   
# cum_input_tokens=0
# cum_output_tokens=0
# name_count=0
# all_name_count=0
for i, row in NOK_input_df.iterrows():
    all_name_count+=1
    image_name = row[IMAGE_COLUMN]
    image_name = str(image_name).strip()
    image_path = dir_images / image_name    

    print(f"Processing • {Path(image_path).stem} • {name_count}/{total_names}", end='\r')   
    try:
        cropped_path = crop_bottom_of_page(image_path, crop_percent=0.3)
        nok = extract_next_of_kin(cropped_path)
        status = "ok"
        name_count+=1
    except Exception as e:
        print("ERROR:", repr(e))
        nok = {
            "next_of_kin_name": "",
            "relationship": "",
            "street_address": "",
            "city": "",
            "state": "",
            "raw_next_of_kin_text": "",
            "uncertain": True
        }
    input_token_cost=.005 # Per 1000
    output_token_cost=.03 # Per 1000
    name_index=Path(image_path).stem
    nok_name=nok.get("next_of_kin_name", "")
    nok_relationship=nok.get("relationship", "")
    
    total_tokens=nok.get("total_tokens", "") 
    input_tokens= nok.get("input_tokens", "")
    output_tokens= nok.get("output_tokens", "")  
    
    cum_input_tokens+=input_tokens
    cum_output_tokens+=output_tokens
    cum_total_tokens+=total_tokens
    cum_cost=f" ${((cum_input_tokens/1000)*input_token_cost+(cum_output_tokens/1000)*output_token_cost):4.2f}"
    
    rows={
        "name_index": Path(image_path).stem,
        "next_of_kin_name": nok.get("next_of_kin_name", ""),
        "relationship": nok.get("relationship", ""),
        "street_address": nok.get("street_address", ""),
        "city": nok.get("city", ""),
        "state": nok.get("state", ""),
        # "raw_next_of_kin_text": nok.get("raw_next_of_kin_text", ""),
        "uncertain": nok.get("uncertain", ""),
        "status": status,
        "input_tokens": nok.get("input_tokens", ""),
        "output_tokens": nok.get("output_tokens", ""),
        "total_tokens": total_tokens,  
        "cum_total_tokens":cum_total_tokens
    }
    NOK_line_df=pd.DataFrame([rows])
    NOK_df = pd.concat([NOK_df, NOK_line_df])
    NOK_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()        
    
    print(f" {blank_line}", end='\r') 
    print(f" Extracted •{name_index} • {nok_name} • {nok_relationship} •• {name_count}/{all_name_count}/{total_names} • Cum Tokens Used:{cum_total_tokens} • Cost: {cum_cost} - Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes") 
print("Done.")
print("Saved:", Path(OUTPUT_CSV).resolve())





**Starting ***  Headstone Next Of Kin (NOK)  *** , Time: 2026-05-07 15:12:42.302088  **

 Extracted •Hearon-William-E-19440923 • Mr. Samuel Hearon • Father •• 142/143/440 • Cum Tokens Used:196340 • Cost:  $1.69 - Process Time:  0.1/ 0.1 Minutes                                 
 Extracted •Hebert-Richard-A-19450424 • Mrs. Rose A. Hebert • Mother •• 143/144/440 • Cum Tokens Used:197717 • Cost:  $1.70 - Process Time:  0.1/ 0.2 Minutes                               
 Extracted •Helbig-Herbert-A-19441130 • Mr. Rudolph Frederick Helbig • Father •• 144/145/440 • Cum Tokens Used:199089 • Cost:  $1.71 - Process Time:  0.1/ 0.3 Minutes                      
 Extracted •Herman-Abraham-P-19441227 • Mrs. Elizabeth Herman • Mother •• 145/146/440 • Cum Tokens Used:200616 • Cost:  $1.73 - Process Time:  0.1/ 0.4 Minutes                             
 Extracted •Hershberger-Kenneth-O-19430329 • Mrs. Virginia Hershberger •  •• 146/147/440 • Cum Tokens Used:201997 • Cost:  $1.74 - Process Time:  0.1/ 0.5 

TypeError: unsupported operand type(s) for +=: 'int' and 'str'

In [93]:
cum_input_tokens*.005/1000

0.020055

In [ ]:
status